# Merchant category and card×category prior history

Exploratory read before adding **per-category amount** features (e.g. z-score vs prior spend in the same `merchant_category` for the same `card_id`).

**Questions:**

- How much volume sits in each category?
- For each row, how many **prior** transactions does this card have **in this category**? (prior-only window, same idea as `prior_tx_count_card` in Gold.)
- What fraction of rows have **sparse** history (`n < 5`, `n < 10`), where one spike could skew mean/std?
- Do sparse buckets differ much by `is_fraud`?

**Spark:** Uses [`config/sparkov.yaml`](../config/sparkov.yaml) Gold path and runtime. If workers use a different Python than the driver, set `PYSPARK_PYTHON` and `PYSPARK_DRIVER_PYTHON` to your `.venv/bin/python`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

project_root = Path.cwd().resolve()
while not (project_root / "src").exists() and project_root.parent != project_root:
    project_root = project_root.parent

src = project_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from fraud_lens.ingest import load_sparkov_config

sparkov_cfg = load_sparkov_config().get("sparkov", {})
gold_path = (project_root / sparkov_cfg.get("gold_path", "data/benchmark/gold_sparkov")).resolve()

spark_builder = SparkSession.builder.appName("FraudLens-CategoryPrior-Explore")
for key, value in sparkov_cfg.get("spark_runtime", {}).items():
    spark_builder = spark_builder.config(key, str(value))
spark = spark_builder.getOrCreate()

print(f"Gold path: {gold_path}")

In [ ]:
base = (
    spark.read.parquet(str(gold_path))
    .where(F.col("is_fraud").isNotNull())
    .select(
        "transaction_id",
        "card_id",
        "merchant_category",
        "amount",
        "event_time_unix",
        "is_fraud",
    )
)

w_card_cat = Window.partitionBy("card_id", "merchant_category").orderBy(
    F.col("event_time_unix"), F.col("transaction_id")
)
w_prior = w_card_cat.rowsBetween(Window.unboundedPreceding, -1)

df = base.withColumn(
    "prior_cat_tx_count", F.count(F.lit(1)).over(w_prior)
).cache()

n_rows = df.count()
print(f"Rows (Sparkov, labeled): {n_rows:,}")

## 1 — Global `merchant_category` volume

In [ ]:
cat_totals = (
    df.groupBy("merchant_category")
    .agg(F.count("*").alias("rows"), F.sum("is_fraud").alias("fraud_rows"))
    .withColumn("fraud_rate", F.round(F.col("fraud_rows") / F.col("rows"), 5))
    .withColumn("row_pct", F.round(100.0 * F.col("rows") / F.lit(n_rows), 3))
    .orderBy(F.desc("rows"))
)

n_cats = cat_totals.count()
print(f"Distinct merchant_category values: {n_cats}")
print("Top 25 categories by row count:")
cat_totals.show(25, truncate=False)

top_pdf = cat_totals.limit(20).toPandas()
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_pdf["merchant_category"][::-1], top_pdf["rows"][::-1], color="#4c78a8")
ax.set_xlabel("transaction rows")
ax.set_title("Top 20 merchant_category by volume")
fig.tight_layout()
plt.show()

## 2 — Prior-only count per row: `(card_id, merchant_category)`

`prior_cat_tx_count` = number of **earlier** rows for the same card in the **same** category (not including the current row).  
First purchase in that category for that card → **0**.

In [ ]:
qs = [0.0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
quantiles = df.approxQuantile("prior_cat_tx_count", qs, relativeError=0.01)
qdf = pd.DataFrame({"q": qs, "prior_cat_tx_count": quantiles})
print("Approximate quantiles of prior_cat_tx_count (all rows):")
print(qdf.to_string(index=False))

frac = df.agg(
    F.avg((F.col("prior_cat_tx_count") == 0).cast("double")).alias("p_n_eq_0"),
    F.avg((F.col("prior_cat_tx_count") == 1).cast("double")).alias("p_n_eq_1"),
    F.avg((F.col("prior_cat_tx_count") < 2).cast("double")).alias("p_n_lt_2"),
    F.avg((F.col("prior_cat_tx_count") < 5).cast("double")).alias("p_n_lt_5"),
    F.avg((F.col("prior_cat_tx_count") < 10).cast("double")).alias("p_n_lt_10"),
    F.avg((F.col("prior_cat_tx_count") < 20).cast("double")).alias("p_n_lt_20"),
).first()

print("\nFraction of rows with sparse prior history in (card, category):")
for k, v in frac.asDict().items():
    print(f"  {k}: {v:.4f}")

In [ ]:
bucket = (
    F.when(F.col("prior_cat_tx_count") == 0, F.lit("0"))
    .when(F.col("prior_cat_tx_count") == 1, F.lit("1"))
    .when(F.col("prior_cat_tx_count") == 2, F.lit("2"))
    .when(F.col("prior_cat_tx_count").between(3, 4), F.lit("3-4"))
    .when(F.col("prior_cat_tx_count").between(5, 9), F.lit("5-9"))
    .when(F.col("prior_cat_tx_count").between(10, 19), F.lit("10-19"))
    .otherwise(F.lit("20+"))
)

hist_df = (
    df.withColumn("prior_bucket", bucket)
    .groupBy("prior_bucket")
    .agg(F.count("*").alias("rows"))
    .withColumn("pct", F.round(100.0 * F.col("rows") / F.lit(n_rows), 2))
).toPandas()

order = ["0", "1", "2", "3-4", "5-9", "10-19", "20+"]
hist_df["prior_bucket"] = pd.Categorical(hist_df["prior_bucket"], categories=order, ordered=True)
hist_df = hist_df.sort_values("prior_bucket")

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(hist_df["prior_bucket"].astype(str), hist_df["rows"], color="#e45756")
ax.set_ylabel("rows")
ax.set_xlabel("prior (card × category) txn count")
ax.set_title("How much history exists before this row in the same category?")
for i, (lab, p) in enumerate(zip(hist_df["prior_bucket"], hist_df["pct"])):
    ax.text(i, hist_df["rows"].iloc[i], f"{p:.1f}%", ha="center", va="bottom", fontsize=8)
fig.tight_layout()
plt.show()

hist_df

## 3 — Sparse history by `is_fraud`

If fraud and non-fraud sit in different **coverage** buckets, card×category z-scores interact differently with the label (still exploratory).

In [ ]:
by_label = (
    df.groupBy("is_fraud")
    .agg(
        F.count("*").alias("rows"),
        F.avg((F.col("prior_cat_tx_count") == 0).cast("double")).alias("p_n_eq_0"),
        F.avg((F.col("prior_cat_tx_count") < 5).cast("double")).alias("p_n_lt_5"),
        F.avg((F.col("prior_cat_tx_count") < 10).cast("double")).alias("p_n_lt_10"),
        F.expr("percentile_approx(prior_cat_tx_count, 0.5)").alias("median_prior_n"),
    )
    .orderBy("is_fraud")
)
by_label.show(truncate=False)

## 4 — Heavy categories: fraction with `prior_cat_tx_count < 5`

For the **top categories by volume**, what share of rows would need a **fallback** if you require at least 5 priors for a stable category-specific baseline?

In [ ]:
top_cats = [r["merchant_category"] for r in cat_totals.select("merchant_category").limit(15).collect()]

sparse_by_cat = (
    df.where(F.col("merchant_category").isin(top_cats))
    .groupBy("merchant_category")
    .agg(
        F.count("*").alias("rows"),
        F.avg((F.col("prior_cat_tx_count") < 5).cast("double")).alias("frac_prior_lt_5"),
        F.avg((F.col("prior_cat_tx_count") < 10).cast("double")).alias("frac_prior_lt_10"),
        F.expr("percentile_approx(prior_cat_tx_count, 0.5)").alias("median_prior_n"),
    )
    .orderBy(F.desc("rows"))
)
print("Top 15 categories: sparse prior history")
sparse_by_cat.show(20, truncate=False)

spdf = sparse_by_cat.toPandas()
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(spdf["merchant_category"][::-1], spdf["frac_prior_lt_5"][::-1], color="#f58518")
ax.set_xlabel("fraction of rows with prior_cat_tx_count < 5")
ax.set_title("Top 15 categories: share needing fallback if K=5")
ax.set_xlim(0, 1)
fig.tight_layout()
plt.show()

## 5 — `log1p(amount)` vs raw `amount` by category (sample)

Quick tail check: heavy-tailed categories stretch linear models; log compresses. Sample **1%** of rows for speed.

In [ ]:
sampled = df.sample(withReplacement=False, fraction=0.01, seed=7).withColumn(
    "log1p_amount", F.log1p(F.col("amount"))
)
sdf = sampled.select("merchant_category", "amount", "log1p_amount").toPandas()

top5 = top_pdf["merchant_category"].head(5).tolist()
sub = sdf[sdf["merchant_category"].isin(top5)]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for cat in top5:
    a = sub.loc[sub["merchant_category"] == cat, "amount"].clip(upper=sub["amount"].quantile(0.99))
    axes[0].hist(np.log1p(a.dropna()), bins=40, alpha=0.35, label=cat)
axes[0].set_title("log1p(amount) within top-5 categories (99th pct clipped raw)")
axes[0].set_xlabel("log1p(amount)")
axes[0].legend(fontsize=7)

axes[1].boxplot(
    [sub.loc[sub["merchant_category"] == c, "log1p_amount"].dropna().values for c in top5],
    tick_labels=top5,
)
axes[1].set_title("log1p(amount) by category (1% sample)")
axes[1].tick_params(axis="x", rotation=25)
fig.tight_layout()
plt.show()

In [ ]:
spark.stop()